In [2]:
import dask.dataframe as dd

# Load the parquets
df_main = dd.read_parquet("transactions_parquet/")
df_time = dd.read_parquet("time_parquet/")

# Print the exact column names
print("Transactions Columns:")
print(list(df_main.columns))

print("\nTime Columns:")
print(list(df_time.columns))

Transactions Columns:
['SHOP_WEEK', 'SHOP_DATE', 'SHOP_WEEKDAY', 'SHOP_HOUR', 'QUANTITY', 'SPEND', 'PROD_CODE', 'PROD_CODE_10', 'PROD_CODE_20', 'PROD_CODE_30', 'PROD_CODE_40', 'CUST_CODE', 'seg_1', 'seg_2', 'BASKET_ID', 'BASKET_SIZE', 'BASKET_PRICE_SENSITIVITY', 'BASKET_TYPE', 'BASKET_DOMINANT_MISSION', 'STORE_CODE', 'STORE_FORMAT', 'STORE_REGION']

Time Columns:
['shop_week', 'date_from', 'date_to']


In [3]:
import dask.dataframe as dd
import pandas as pd

print("1. Loading Parquet files lazily...")
df_main = dd.read_parquet("transactions_parquet/")
df_time = dd.read_parquet("time_parquet/")

print("2. Fixing column names for a perfect merge...")
# Rename 'shop_week' in the time dataset to match the main dataset exactly
df_time = df_time.rename(columns={'shop_week': 'SHOP_WEEK'})

print("3. Building the Cleaning & Merging Pipeline...")
# Now they match perfectly, so the merge will succeed
df_merged = dd.merge(df_main, df_time, on='SHOP_WEEK', how='left')

# Drop invalid rows using your EXACT column names
# 'CUST_CODE' is the shopper, 'SPEND' is the revenue
df_clean = df_merged.dropna(subset=['CUST_CODE', 'SPEND'])

# Filter out returns (negative quantity) and zero-dollar items
df_clean = df_clean[(df_clean['QUANTITY'] > 0) & (df_clean['SPEND'] > 0)]

# Convert your SHOP_DATE column to a proper datetime format
# Dask requires a slightly different datetime conversion method
df_clean['Transaction_Date'] = dd.to_datetime(df_clean['SHOP_DATE'], format='%Y%m%d', errors='coerce')

print("4. Executing the Pipeline (This may take a minute)...")
preview = df_clean.head(10)

print("-" * 50)
print("PIPELINE SUCCESSFUL! HERE IS YOUR CLEAN DATA:")
print("-" * 50)
display(preview)

1. Loading Parquet files lazily...
2. Fixing column names for a perfect merge...
3. Building the Cleaning & Merging Pipeline...
4. Executing the Pipeline (This may take a minute)...
--------------------------------------------------
PIPELINE SUCCESSFUL! HERE IS YOUR CLEAN DATA:
--------------------------------------------------


,SHOP_WEEK,SHOP_DATE,SHOP_WEEKDAY,SHOP_HOUR,QUANTITY,SPEND,PROD_CODE,PROD_CODE_10,PROD_CODE_20,PROD_CODE_30,...,BASKET_SIZE,BASKET_PRICE_SENSITIVITY,BASKET_TYPE,BASKET_DOMINANT_MISSION,STORE_CODE,STORE_FORMAT,STORE_REGION,date_from,date_to,Transaction_Date
0,200607,20060410,2,16,1,1.01,PRD0900005,CL00155,DEP00053,G00016,...,L,MM,Top Up,Mixed,STORE00001,LS,E02,20060410,20060416,2006-04-10
1,200607,20060416,1,9,3,3.03,PRD0900005,CL00155,DEP00053,G00016,...,L,UM,Top Up,Fresh,STORE00001,LS,E02,20060410,20060416,2006-04-16
2,200607,20060413,5,19,1,0.22,PRD0900006,CL00103,DEP00035,G00010,...,L,MM,Full Shop,Mixed,STORE00001,LS,E02,20060410,20060416,2006-04-13
3,200607,20060416,1,17,1,0.22,PRD0900007,CL00103,DEP00035,G00010,...,L,MM,Full Shop,Fresh,STORE00001,LS,E02,20060410,20060416,2006-04-16
4,200607,20060414,6,17,1,1.77,PRD0900008,CL00042,DEP00011,G00004,...,L,UM,Full Shop,Mixed,STORE00001,LS,E02,20060410,20060416,2006-04-14
6,200607,20060411,3,14,1,0.92,PRD0900014,CL00089,DEP00025,G00008,...,L,LA,Full Shop,Fresh,STORE00001,LS,E02,20060410,20060416,2006-04-11
7,200607,20060414,6,17,1,1.99,PRD0900017,CL00140,DEP00049,G00014,...,L,UM,Full Shop,Mixed,STORE00001,LS,E02,20060410,20060416,2006-04-14
8,200607,20060414,6,16,1,1.48,PRD0900021,CL00006,DEP00002,G00001,...,L,MM,Top Up,Fresh,STORE00001,LS,E02,20060410,20060416,2006-04-14
9,200607,20060411,3,20,1,1.38,PRD0900026,CL00144,DEP00051,G00015,...,S,MM,Small Shop,Mixed,STORE00001,LS,E02,20060410,20060416,2006-04-11
10,200607,20060414,6,15,3,5.61,PRD0900027,CL00074,DEP00021,G00007,...,L,MM,Full Shop,Fresh,STORE00001,LS,E02,20060410,20060416,2006-04-14


In [4]:
import dask.dataframe as dd

print("--- STEP 1: MEMORY-OPTIMIZED BASIC CLEANING ---")
print("1. Loading datasets...")
df_main = dd.read_parquet("transactions_parquet/")
df_time = dd.read_parquet("time_parquet/")

print("2. Merging...")
df_time = df_time.rename(columns={'shop_week': 'SHOP_WEEK'})
df = dd.merge(df_main, df_time, on='SHOP_WEEK', how='left')

print("3. Executing Memory-Safe Cleaning...")

# A. Drop nulls FIRST. This instantly shrinks the data footprint in memory.
critical_cols = ['CUST_CODE', 'SPEND', 'QUANTITY', 'SHOP_DATE']
df_clean = df.dropna(subset=critical_cols)

# B. Remove returns and zero-dollar items
df_clean = df_clean[(df_clean['QUANTITY'] > 0) & (df_clean['SPEND'] > 0)]

# C. Parse the dates
df_clean['Transaction_Date'] = dd.to_datetime(df_clean['SHOP_DATE'], format='%Y%m%d', errors='coerce')
df_clean = df_clean.dropna(subset=['Transaction_Date'])

print("4. Computing final count (Optimized)...")
# map_partitions is a much safer, memory-friendly way to count massive Dask dataframes
final_rows = df_clean.map_partitions(len).compute().sum()

print("-" * 50)
print("CLEANING COMPLETE")
print(f"Total valid transactions remaining: {final_rows:,}")
print("-" * 50)

display(df_clean.head(5))

--- STEP 1: MEMORY-OPTIMIZED BASIC CLEANING ---
1. Loading datasets...
2. Merging...
3. Executing Memory-Safe Cleaning...
4. Computing final count (Optimized)...
--------------------------------------------------
CLEANING COMPLETE
Total valid transactions remaining: 247,997,003
--------------------------------------------------


,SHOP_WEEK,SHOP_DATE,SHOP_WEEKDAY,SHOP_HOUR,QUANTITY,SPEND,PROD_CODE,PROD_CODE_10,PROD_CODE_20,PROD_CODE_30,...,BASKET_SIZE,BASKET_PRICE_SENSITIVITY,BASKET_TYPE,BASKET_DOMINANT_MISSION,STORE_CODE,STORE_FORMAT,STORE_REGION,date_from,date_to,Transaction_Date
0,200607,20060410,2,16,1,1.01,PRD0900005,CL00155,DEP00053,G00016,...,L,MM,Top Up,Mixed,STORE00001,LS,E02,20060410,20060416,2006-04-10
1,200607,20060416,1,9,3,3.03,PRD0900005,CL00155,DEP00053,G00016,...,L,UM,Top Up,Fresh,STORE00001,LS,E02,20060410,20060416,2006-04-16
2,200607,20060413,5,19,1,0.22,PRD0900006,CL00103,DEP00035,G00010,...,L,MM,Full Shop,Mixed,STORE00001,LS,E02,20060410,20060416,2006-04-13
3,200607,20060416,1,17,1,0.22,PRD0900007,CL00103,DEP00035,G00010,...,L,MM,Full Shop,Fresh,STORE00001,LS,E02,20060410,20060416,2006-04-16
4,200607,20060414,6,17,1,1.77,PRD0900008,CL00042,DEP00011,G00004,...,L,UM,Full Shop,Mixed,STORE00001,LS,E02,20060410,20060416,2006-04-14


In [5]:
# --- STEP 2: DEEP CLEANING & TIMELINE EXTRACTION ---
print("1. Calculating Outlier Thresholds (99th Percentile)...")

# Calculate the 99th percentile limits to cut out B2B/Wholesale buyers
spend_99 = df_clean['SPEND'].quantile(0.99).compute()
qty_99 = df_clean['QUANTITY'].quantile(0.99).compute()

print(f"   -> Max Allowable Spend per item: {spend_99:.2f}")
print(f"   -> Max Allowable Quantity per item: {qty_99:.2f}")

print("\n2. Applying Deep Clean Filter...")
# Keep only standard consumer behavior
df_final_clean = df_clean[(df_clean['SPEND'] <= spend_99) & (df_clean['QUANTITY'] <= qty_99)]

# Count the final, perfectly clean dataset
final_rows_count = df_final_clean.map_partitions(len).compute().sum()
print(f"   -> Final Cleaned Rows Remaining: {final_rows_count:,}")

print("\n3. Extracting the Timeline...")
# Scan the massive dataset to find our exact start and end dates
min_date = df_final_clean['Transaction_Date'].min().compute()
max_date = df_final_clean['Transaction_Date'].max().compute()

print("-" * 50)
print("DATASET PREPARATION COMPLETE")
print(f"First Transaction: {min_date}")
print(f"Last Transaction:  {max_date}")
print("-" * 50)

1. Calculating Outlier Thresholds (99th Percentile)...
   -> Max Allowable Spend per item: 16.29
   -> Max Allowable Quantity per item: 10.00

2. Applying Deep Clean Filter...
   -> Final Cleaned Rows Remaining: 245,976,814

3. Extracting the Timeline...
--------------------------------------------------
DATASET PREPARATION COMPLETE
First Transaction: 2006-04-10 00:00:00
Last Transaction:  2008-07-06 00:00:00
--------------------------------------------------


In [6]:
# --- STEP 3: FEATURE SELECTION & FINAL NULL CHECK ---
print("1. Dropping Unnecessary Categorical Data...")

# For our VIP Retention Model, we only need the exact columns that prove 
# WHO bought it, WHEN they bought it, WHAT they paid, and HOW DIVERSE their basket was.
# This automatically drops all the noisy store and region categories.
core_columns = [
    'CUST_CODE',          # The 'Who'
    'Transaction_Date',   # The 'When'
    'SPEND',              # Monetary
    'QUANTITY',           # Frequency / Volume
    'PROD_CODE',          # Product Diversity
    'BASKET_ID'           # To track unique visits
]

df_model = df_final_clean[core_columns]

print("2. Scanning for any remaining missing values in our core features...")
# Dask will scan the entire remaining dataset to ensure no nulls snuck through
missing_values = df_model.isnull().sum().compute()
print("\nMissing Values Count:")
print(missing_values)

print("\n3. Finalizing the Pristine Dataset...")
# Just in case the scan found anything, we drop it here to be perfectly safe
df_model = df_model.dropna()

print("-" * 50)
print("FINAL DATA CLEANING COMPLETE")
print("Dataset is now memory-optimized and mathematically pure.")
print("-" * 50)

# Display the streamlined dataset
display(df_model.head())

1. Dropping Unnecessary Categorical Data...
2. Scanning for any remaining missing values in our core features...

Missing Values Count:
CUST_CODE           0
Transaction_Date    0
SPEND               0
QUANTITY            0
PROD_CODE           0
BASKET_ID           0
dtype: int64

3. Finalizing the Pristine Dataset...
--------------------------------------------------
FINAL DATA CLEANING COMPLETE
Dataset is now memory-optimized and mathematically pure.
--------------------------------------------------


,CUST_CODE,Transaction_Date,SPEND,QUANTITY,PROD_CODE,BASKET_ID
0,CUST0000167145,2006-04-10,1.01,1,PRD0900005,994100100253028
1,CUST0000246027,2006-04-16,3.03,3,PRD0900005,994100100299966
2,CUST0000140993,2006-04-13,0.22,1,PRD0900006,994100100237197
3,CUST0000938246,2006-04-16,0.22,1,PRD0900007,994100100715506
4,CUST0000994485,2006-04-14,1.77,1,PRD0900008,994100100749139


In [7]:
import pandas as pd
import datetime
import dask

# --- STEP 4: DYNAMIC TIME SLICING & FEATURE ENGINEERING (DASK 'NUNIQUE' FIX) ---
print("1. Automatically detecting the Holiday Window...")

max_date = df_model['Transaction_Date'].max().compute()
target_year = max_date.year if max_date.month > 6 else max_date.year - 1

holiday_start = pd.to_datetime(f"{target_year}-11-01")
holiday_end = pd.to_datetime(f"{target_year}-12-31")
post_start = pd.to_datetime(f"{target_year + 1}-01-01")
post_end = pd.to_datetime(f"{target_year + 1}-06-30")

print(f"   -> Holiday Peak Assigned: {holiday_start.date()} to {holiday_end.date()}")
print(f"   -> Post-Holiday Tracking: {post_start.date()} to {post_end.date()}")

print("\n2. Slicing the Dask Dataframes...")
holiday_df = df_model[(df_model['Transaction_Date'] >= holiday_start) & 
                      (df_model['Transaction_Date'] <= holiday_end)]

post_holiday_df = df_model[(df_model['Transaction_Date'] >= post_start) & 
                           (df_model['Transaction_Date'] <= post_end)]

pre_holiday_df = df_model[df_model['Transaction_Date'] < holiday_start]

print("3. Engineering RFM & Diversity Features (Bypassing Dask limit with parallel compute)...")

# --- A. Holiday Features ---
# FIX: We define the aggregations separately to avoid the dictionary 'nunique' error
h_group = holiday_df.groupby('CUST_CODE')

h_recency = h_group['Transaction_Date'].max()
h_freq = h_group['BASKET_ID'].nunique()
h_monetary = h_group['SPEND'].sum()
h_unique_prods = h_group['PROD_CODE'].nunique()
h_qty = h_group['QUANTITY'].sum()

print("   -> Computing Holiday metrics...")
# Compute them all at once into memory (Highly efficient!)
h_recency, h_freq, h_monetary, h_unique_prods, h_qty = dask.compute(
    h_recency, h_freq, h_monetary, h_unique_prods, h_qty
)

# Combine into a standard Pandas DataFrame
holiday_features = pd.concat([
    h_recency.rename('Last_Purchase_Date'),
    h_freq.rename('Holiday_Frequency'),
    h_monetary.rename('Holiday_Monetary'),
    h_unique_prods.rename('Holiday_UniqueProducts'),
    h_qty.rename('Holiday_TotalItems')
], axis=1)

# Force the date column to be an absolute datetime format, turning errors into NaT
holiday_features['Last_Purchase_Date'] = pd.to_datetime(holiday_features['Last_Purchase_Date'], errors='coerce')
holiday_features = holiday_features.dropna(subset=['Last_Purchase_Date'])

# Calculate Recency and Product Diversity
holiday_features['Holiday_Recency'] = (holiday_end - holiday_features['Last_Purchase_Date']).dt.days
holiday_features['Holiday_ProductDiversity'] = (
    holiday_features['Holiday_UniqueProducts'] / holiday_features['Holiday_TotalItems'].clip(lower=1)
)

# --- B. Pre-Holiday Baseline ---
print("   -> Computing Pre-Holiday metrics...")
pre_group = pre_holiday_df.groupby('CUST_CODE')

pre_freq = pre_group['BASKET_ID'].nunique()
pre_monetary = pre_group['SPEND'].sum()

pre_freq, pre_monetary = dask.compute(pre_freq, pre_monetary)

pre_features = pd.concat([
    pre_freq.rename('Pre_Frequency'),
    pre_monetary.rename('Pre_Monetary')
], axis=1)

# --- C. Target Variable ---
print("   -> Computing Post-Holiday retention status...")
retained_customers = post_holiday_df['CUST_CODE'].unique().compute()

print("\n4. Assembling the Final Machine Learning Dataset...")
ml_data = holiday_features.copy()
ml_data = ml_data.join(pre_features, how='left') 
ml_data = ml_data.fillna(0) 

ml_data['Retained'] = ml_data.index.isin(retained_customers).astype(int)

# Drop raw calculation columns
ml_data = ml_data.drop(columns=['Last_Purchase_Date', 'Holiday_UniqueProducts', 'Holiday_TotalItems'])

print("-" * 50)
print("FEATURE ENGINEERING COMPLETE!")
print(f"Total Holiday Shoppers to Analyze: {len(ml_data):,}")
print(f"Overall Retained (1): {ml_data['Retained'].sum():,}")
print(f"Overall Churned (0):  {len(ml_data) - ml_data['Retained'].sum():,}")
print("-" * 50)

display(ml_data.head())

1. Automatically detecting the Holiday Window...
   -> Holiday Peak Assigned: 2008-11-01 to 2008-12-31
   -> Post-Holiday Tracking: 2009-01-01 to 2009-06-30

2. Slicing the Dask Dataframes...
3. Engineering RFM & Diversity Features (Bypassing Dask limit with parallel compute)...
   -> Computing Holiday metrics...
   -> Computing Pre-Holiday metrics...
   -> Computing Post-Holiday retention status...

4. Assembling the Final Machine Learning Dataset...
--------------------------------------------------
FEATURE ENGINEERING COMPLETE!
Total Holiday Shoppers to Analyze: 0
Overall Retained (1): 0
Overall Churned (0):  0
--------------------------------------------------


,Holiday_Frequency,Holiday_Monetary,Holiday_Recency,Holiday_ProductDiversity,Pre_Frequency,Pre_Monetary,Retained
CUST_CODE,,,,,,,


In [8]:
import pandas as pd
import datetime
import dask

# --- STEP 4: DYNAMIC TIME SLICING & FEATURE ENGINEERING (MERGE FIX) ---
print("1. Automatically detecting the Holiday Window...")

max_date = df_model['Transaction_Date'].max().compute()
target_year = max_date.year if max_date.month > 6 else max_date.year - 1

holiday_start = pd.to_datetime(f"{target_year}-11-01")
holiday_end = pd.to_datetime(f"{target_year}-12-31")
post_start = pd.to_datetime(f"{target_year + 1}-01-01")
post_end = pd.to_datetime(f"{target_year + 1}-06-30")

print(f"   -> Holiday Peak Assigned: {holiday_start.date()} to {holiday_end.date()}")
print(f"   -> Post-Holiday Tracking: {post_start.date()} to {post_end.date()}")

print("\n2. Slicing the Dask Dataframes...")
holiday_df = df_model[(df_model['Transaction_Date'] >= holiday_start) & 
                      (df_model['Transaction_Date'] <= holiday_end)]

post_holiday_df = df_model[(df_model['Transaction_Date'] >= post_start) & 
                           (df_model['Transaction_Date'] <= post_end)]

pre_holiday_df = df_model[df_model['Transaction_Date'] < holiday_start]

print("3. Engineering RFM & Diversity Features (Executing parallel compute)...")

# --- A. Holiday Features ---
h_group = holiday_df.groupby('CUST_CODE')

h_recency = h_group['Transaction_Date'].max()
h_freq = h_group['BASKET_ID'].nunique()
h_monetary = h_group['SPEND'].sum()
h_unique_prods = h_group['PROD_CODE'].nunique()
h_qty = h_group['QUANTITY'].sum()

print("   -> Computing Holiday metrics...")
h_recency, h_freq, h_monetary, h_unique_prods, h_qty = dask.compute(
    h_recency, h_freq, h_monetary, h_unique_prods, h_qty
)

# THE FIX: We use .reset_index() to force CUST_CODE into a real, standard column
holiday_features = pd.concat([
    h_recency.rename('Last_Purchase_Date'),
    h_freq.rename('Holiday_Frequency'),
    h_monetary.rename('Holiday_Monetary'),
    h_unique_prods.rename('Holiday_UniqueProducts'),
    h_qty.rename('Holiday_TotalItems')
], axis=1).reset_index() 

holiday_features['Last_Purchase_Date'] = pd.to_datetime(holiday_features['Last_Purchase_Date'], errors='coerce')
holiday_features = holiday_features.dropna(subset=['Last_Purchase_Date'])

holiday_features['Holiday_Recency'] = (holiday_end - holiday_features['Last_Purchase_Date']).dt.days
holiday_features['Holiday_ProductDiversity'] = (
    holiday_features['Holiday_UniqueProducts'] / holiday_features['Holiday_TotalItems'].clip(lower=1)
)

# --- B. Pre-Holiday Baseline ---
print("   -> Computing Pre-Holiday metrics...")
pre_group = pre_holiday_df.groupby('CUST_CODE')

pre_freq = pre_group['BASKET_ID'].nunique()
pre_monetary = pre_group['SPEND'].sum()

pre_freq, pre_monetary = dask.compute(pre_freq, pre_monetary)

# THE FIX: Force Pre-Holiday CUST_CODE into a standard column as well
pre_features = pd.concat([
    pre_freq.rename('Pre_Frequency'),
    pre_monetary.rename('Pre_Monetary')
], axis=1).reset_index() 

# --- C. Target Variable ---
print("   -> Computing Post-Holiday retention status...")
retained_customers = post_holiday_df['CUST_CODE'].unique().compute()

print("\n4. Assembling the Final Machine Learning Dataset...")

# THE FIX: Merge safely using the explicit column name instead of hidden indices
ml_data = holiday_features.merge(pre_features, on='CUST_CODE', how='left')
ml_data = ml_data.fillna(0) 

# Check retention based on the explicit column
ml_data['Retained'] = ml_data['CUST_CODE'].isin(retained_customers).astype(int)

# Clean up raw columns and set CUST_CODE back as the index for Machine Learning
ml_data = ml_data.drop(columns=['Last_Purchase_Date', 'Holiday_UniqueProducts', 'Holiday_TotalItems'])
ml_data = ml_data.set_index('CUST_CODE')

print("-" * 50)
print("FEATURE ENGINEERING COMPLETE!")
print(f"Total Holiday Shoppers to Analyze: {len(ml_data):,}")
print(f"Overall Retained (1): {ml_data['Retained'].sum():,}")
print(f"Overall Churned (0):  {len(ml_data) - ml_data['Retained'].sum():,}")
print("-" * 50)

display(ml_data.head())

1. Automatically detecting the Holiday Window...
   -> Holiday Peak Assigned: 2008-11-01 to 2008-12-31
   -> Post-Holiday Tracking: 2009-01-01 to 2009-06-30

2. Slicing the Dask Dataframes...
3. Engineering RFM & Diversity Features (Executing parallel compute)...
   -> Computing Holiday metrics...
   -> Computing Pre-Holiday metrics...
   -> Computing Post-Holiday retention status...

4. Assembling the Final Machine Learning Dataset...
--------------------------------------------------
FEATURE ENGINEERING COMPLETE!
Total Holiday Shoppers to Analyze: 0
Overall Retained (1): 0
Overall Churned (0):  0
--------------------------------------------------


,Holiday_Frequency,Holiday_Monetary,Holiday_Recency,Holiday_ProductDiversity,Pre_Frequency,Pre_Monetary,Retained
CUST_CODE,,,,,,,


In [9]:
import pandas as pd
import datetime
import dask

# --- STEP 4: DYNAMIC TIME SLICING (DATE LOGIC FIX) ---
print("1. Automatically detecting a SAFE Holiday Window...")

max_date = df_model['Transaction_Date'].max().compute()

# THE FIX: We need a full 6 months AFTER Christmas to track retention.
# If we don't have up to June 30th of the current max year, we must use the Christmas from TWO years ago.
if max_date >= pd.to_datetime(f"{max_date.year}-06-30"):
    target_year = max_date.year - 1
else:
    target_year = max_date.year - 2

holiday_start = pd.to_datetime(f"{target_year}-11-01")
holiday_end = pd.to_datetime(f"{target_year}-12-31")
post_start = pd.to_datetime(f"{target_year + 1}-01-01")
post_end = pd.to_datetime(f"{target_year + 1}-06-30")

print(f"   -> Dataset Ends: {max_date.date()}")
print(f"   -> Safe Holiday Peak Assigned: {holiday_start.date()} to {holiday_end.date()}")
print(f"   -> Post-Holiday Tracking: {post_start.date()} to {post_end.date()}")

print("\n2. Slicing the Dask Dataframes...")
holiday_df = df_model[(df_model['Transaction_Date'] >= holiday_start) & 
                      (df_model['Transaction_Date'] <= holiday_end)]

post_holiday_df = df_model[(df_model['Transaction_Date'] >= post_start) & 
                           (df_model['Transaction_Date'] <= post_end)]

pre_holiday_df = df_model[df_model['Transaction_Date'] < holiday_start]

print("3. Engineering RFM & Diversity Features (Executing parallel compute)...")

# --- A. Holiday Features ---
h_group = holiday_df.groupby('CUST_CODE')

h_recency = h_group['Transaction_Date'].max()
h_freq = h_group['BASKET_ID'].nunique()
h_monetary = h_group['SPEND'].sum()
h_unique_prods = h_group['PROD_CODE'].nunique()
h_qty = h_group['QUANTITY'].sum()

print("   -> Computing Holiday metrics...")
h_recency, h_freq, h_monetary, h_unique_prods, h_qty = dask.compute(
    h_recency, h_freq, h_monetary, h_unique_prods, h_qty
)

holiday_features = pd.concat([
    h_recency.rename('Last_Purchase_Date'),
    h_freq.rename('Holiday_Frequency'),
    h_monetary.rename('Holiday_Monetary'),
    h_unique_prods.rename('Holiday_UniqueProducts'),
    h_qty.rename('Holiday_TotalItems')
], axis=1).reset_index() 

holiday_features['Last_Purchase_Date'] = pd.to_datetime(holiday_features['Last_Purchase_Date'], errors='coerce')
holiday_features = holiday_features.dropna(subset=['Last_Purchase_Date'])

holiday_features['Holiday_Recency'] = (holiday_end - holiday_features['Last_Purchase_Date']).dt.days
holiday_features['Holiday_ProductDiversity'] = (
    holiday_features['Holiday_UniqueProducts'] / holiday_features['Holiday_TotalItems'].clip(lower=1)
)

# --- B. Pre-Holiday Baseline ---
print("   -> Computing Pre-Holiday metrics...")
pre_group = pre_holiday_df.groupby('CUST_CODE')

pre_freq = pre_group['BASKET_ID'].nunique()
pre_monetary = pre_group['SPEND'].sum()

pre_freq, pre_monetary = dask.compute(pre_freq, pre_monetary)

pre_features = pd.concat([
    pre_freq.rename('Pre_Frequency'),
    pre_monetary.rename('Pre_Monetary')
], axis=1).reset_index() 

# --- C. Target Variable ---
print("   -> Computing Post-Holiday retention status...")
retained_customers = post_holiday_df['CUST_CODE'].unique().compute()

print("\n4. Assembling the Final Machine Learning Dataset...")
ml_data = holiday_features.merge(pre_features, on='CUST_CODE', how='left')
ml_data = ml_data.fillna(0) 

ml_data['Retained'] = ml_data['CUST_CODE'].isin(retained_customers).astype(int)

ml_data = ml_data.drop(columns=['Last_Purchase_Date', 'Holiday_UniqueProducts', 'Holiday_TotalItems'])
ml_data = ml_data.set_index('CUST_CODE')

print("-" * 50)
print("FEATURE ENGINEERING COMPLETE!")
print(f"Total Holiday Shoppers to Analyze: {len(ml_data):,}")
print(f"Overall Retained (1): {ml_data['Retained'].sum():,}")
print(f"Overall Churned (0):  {len(ml_data) - ml_data['Retained'].sum():,}")
print("-" * 50)

display(ml_data.head())

1. Automatically detecting a SAFE Holiday Window...
   -> Dataset Ends: 2008-07-06
   -> Safe Holiday Peak Assigned: 2007-11-01 to 2007-12-31
   -> Post-Holiday Tracking: 2008-01-01 to 2008-06-30

2. Slicing the Dask Dataframes...
3. Engineering RFM & Diversity Features (Executing parallel compute)...
   -> Computing Holiday metrics...
   -> Computing Pre-Holiday metrics...
   -> Computing Post-Holiday retention status...

4. Assembling the Final Machine Learning Dataset...
--------------------------------------------------
FEATURE ENGINEERING COMPLETE!
Total Holiday Shoppers to Analyze: 309,415
Overall Retained (1): 298,367
Overall Churned (0):  11,048
--------------------------------------------------


,Holiday_Frequency,Holiday_Monetary,Holiday_Recency,Holiday_ProductDiversity,Pre_Frequency,Pre_Monetary,Retained
CUST_CODE,,,,,,,
CUST0000000327,29,196.01,0,0.408163,252.0,1148.77,1
CUST0000001173,22,129.84,3,0.543046,312.0,2221.38,1
CUST0000001247,13,35.63,2,0.642857,150.0,1070.39,1
CUST0000001981,6,20.52,33,0.473684,54.0,110.70,1
CUST0000002098,3,96.54,12,0.596774,18.0,835.41,1


In [10]:
# --- STEP 5: GMM CLUSTERING TO FIND TRUE VIPS ---
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

print("1. Preparing data for Clustering...")
# We strictly use Holiday behaviors to define Holiday VIPs (No data leakage!)
cluster_features = ['Holiday_Recency', 'Holiday_Frequency', 'Holiday_Monetary']
X_cluster = ml_data[cluster_features]

print("2. Scaling the features (Crucial for GMM math)...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

print("3. Applying Gaussian Mixture Model (3 Clusters)...")
print("   (Please wait, processing 300,000+ rows might take a minute!)")
# We divide the shoppers into 3 mathematical distributions
gmm = GaussianMixture(n_components=3, random_state=42, covariance_type='full')
ml_data['Cluster'] = gmm.fit_predict(X_scaled)

print("4. Identifying the VIP Segment...")
# The true VIP cluster will naturally have the highest average monetary value
cluster_summary = ml_data.groupby('Cluster')[['Holiday_Monetary', 'Holiday_Frequency', 'Holiday_Recency']].mean()
vip_cluster_id = cluster_summary['Holiday_Monetary'].idxmax()

# Slice the dataset to isolate ONLY the VIPs
vip_data = ml_data[ml_data['Cluster'] == vip_cluster_id].copy()

print("\n" + "="*50)
print("GMM CLUSTERING COMPLETE")
print("="*50)
print("Cluster Averages (Look for the massive difference in Spend/Frequency!):")
print(cluster_summary.round(2))
print("-" * 50)
print(f"Identified Cluster {vip_cluster_id} as the VIP Segment.")
print(f"Total True VIPs Found: {len(vip_data):,}")
print(f"VIPs Retained (1): {vip_data['Retained'].sum():,}")
print(f"VIPs Churned (0):  {len(vip_data) - vip_data['Retained'].sum():,}")
print("="*50)

display(vip_data.head())

1. Preparing data for Clustering...
2. Scaling the features (Crucial for GMM math)...
3. Applying Gaussian Mixture Model (3 Clusters)...
   (Please wait, processing 300,000+ rows might take a minute!)
4. Identifying the VIP Segment...

GMM CLUSTERING COMPLETE
Cluster Averages (Look for the massive difference in Spend/Frequency!):
         Holiday_Monetary  Holiday_Frequency  Holiday_Recency
Cluster                                                      
0                   89.58               7.72             8.58
1                  247.04              19.46             3.06
2                   17.84               2.07            24.16
--------------------------------------------------
Identified Cluster 1 as the VIP Segment.
Total True VIPs Found: 80,908
VIPs Retained (1): 80,898
VIPs Churned (0):  10


,Holiday_Frequency,Holiday_Monetary,Holiday_Recency,Holiday_ProductDiversity,Pre_Frequency,Pre_Monetary,Retained,Cluster
CUST_CODE,,,,,,,,
CUST0000000327,29,196.01,0,0.408163,252.0,1148.77,1,1
CUST0000001173,22,129.84,3,0.543046,312.0,2221.38,1,1
CUST0000007796,18,66.82,1,0.820513,158.0,520.29,1,1
CUST0000008102,8,499.49,10,0.264706,160.0,4738.36,1,1
CUST0000009151,34,613.80,0,0.239180,144.0,2886.59,1,1


In [11]:
# --- STEP 5B: FIXING IMBALANCE VIA 'VALUE CHURN' ---
print("1. Extracting Post-Holiday Spending for our VIPs (Executing Dask compute)...")

# Get the unique list of our 80k VIPs
vip_list = vip_data.index.tolist()

# Calculate their total spend in the 6 months AFTER the holidays
# We filter the Dask dataframe to only include VIPs to save memory
post_vip_df = post_holiday_df[post_holiday_df['CUST_CODE'].isin(vip_list)]
post_spend = post_vip_df.groupby('CUST_CODE')['SPEND'].sum().compute()

print("2. Redefining the Target Variable (Retaining VIP Status)...")
# Map their post-holiday spend back to our Pandas dataframe. Fill 0 if they vanished entirely.
vip_data['Post_Holiday_Spend'] = vip_data.index.map(post_spend).fillna(0)

# Calculate Monthly Run-Rates to make a fair comparison
# Holiday Peak was exactly 2 months (Nov 1 to Dec 31)
vip_data['Holiday_Monthly_RunRate'] = vip_data['Holiday_Monetary'] / 2

# Post-Holiday Tracking was exactly 6 months (Jan 1 to Jun 30)
vip_data['Post_Monthly_RunRate'] = vip_data['Post_Holiday_Spend'] / 6

# THE NEW TARGET LOGIC:
# To "Retain VIP Status", a customer must maintain at least 50% of their peak holiday monthly spend.
# If their monthly spend drops by more than half, they have "Downgraded" (Churned from the VIP tier).
vip_data['Retained'] = (vip_data['Post_Monthly_RunRate'] >= (vip_data['Holiday_Monthly_RunRate'] * 0.5)).astype(int)

# Clean up the temporary calculation columns so the data is ready for modeling
vip_data = vip_data.drop(columns=['Post_Holiday_Spend', 'Holiday_Monthly_RunRate', 'Post_Monthly_RunRate'])

print("\n" + "="*50)
print("NEW TARGET VARIABLE DISTRIBUTION ('VALUE CHURN')")
print("="*50)
print(f"Maintained VIP Status (1): {vip_data['Retained'].sum():,}")
print(f"Downgraded / Lost VIP Status (0): {len(vip_data) - vip_data['Retained'].sum():,}")
print("="*50)

# Preview the final, perfectly prepped dataset
display(vip_data.head())

1. Extracting Post-Holiday Spending for our VIPs (Executing Dask compute)...
2. Redefining the Target Variable (Retaining VIP Status)...

NEW TARGET VARIABLE DISTRIBUTION ('VALUE CHURN')
Maintained VIP Status (1): 74,615
Downgraded / Lost VIP Status (0): 6,293


,Holiday_Frequency,Holiday_Monetary,Holiday_Recency,Holiday_ProductDiversity,Pre_Frequency,Pre_Monetary,Retained,Cluster
CUST_CODE,,,,,,,,
CUST0000000327,29,196.01,0,0.408163,252.0,1148.77,1,1
CUST0000001173,22,129.84,3,0.543046,312.0,2221.38,1,1
CUST0000007796,18,66.82,1,0.820513,158.0,520.29,1,1
CUST0000008102,8,499.49,10,0.264706,160.0,4738.36,1,1
CUST0000009151,34,613.80,0,0.239180,144.0,2886.59,1,1


In [13]:
# --- EXPLORING NATURAL BALANCE VIA STRICTER THRESHOLDS (FIXED) ---
print("Testing Stricter VIP Retention Thresholds...\n")

# THE FIX: Bring back the Post_Holiday_Spend data that we dropped in the last step
if 'Post_Holiday_Spend' not in vip_data.columns:
    vip_data['Post_Holiday_Spend'] = vip_data.index.map(post_spend).fillna(0)

# Calculate original run rates
vip_data['Holiday_Monthly_RunRate'] = vip_data['Holiday_Monetary'] / 2
vip_data['Post_Monthly_RunRate'] = vip_data['Post_Holiday_Spend'] / 6

# Test 1: The 70% Rule (Must maintain 70% of Holiday Spend)
vip_data['Retained_70'] = (vip_data['Post_Monthly_RunRate'] >= (vip_data['Holiday_Monthly_RunRate'] * 0.70)).astype(int)

# Test 2: The 85% Rule (Must maintain 85% of Holiday Spend)
vip_data['Retained_85'] = (vip_data['Post_Monthly_RunRate'] >= (vip_data['Holiday_Monthly_RunRate'] * 0.85)).astype(int)

print("Scenario A: The 70% Maintenance Rule")
print(f"Maintained (1): {vip_data['Retained_70'].sum():,}")
print(f"Downgraded (0): {len(vip_data) - vip_data['Retained_70'].sum():,}")
print("-" * 40)

print("Scenario B: The 85% Maintenance Rule")
print(f"Maintained (1): {vip_data['Retained_85'].sum():,}")
print(f"Downgraded (0): {len(vip_data) - vip_data['Retained_85'].sum():,}")

Testing Stricter VIP Retention Thresholds...

Scenario A: The 70% Maintenance Rule
Maintained (1): 60,259
Downgraded (0): 20,649
----------------------------------------
Scenario B: The 85% Maintenance Rule
Maintained (1): 42,841
Downgraded (0): 38,067


In [15]:
# --- DATA PREPARATION: LOCKING IN THE BAKE-OFF DATA ---
from sklearn.model_selection import train_test_split

print("1. Finalizing Target Variable (Using your 70% Rule choice)...")
vip_data['Retained'] = vip_data['Retained_70']

print("2. Isolating the Features...")
features = ['Holiday_Recency', 'Holiday_Frequency', 'Holiday_Monetary', 
            'Holiday_ProductDiversity', 'Pre_Frequency', 'Pre_Monetary']

X = vip_data[features]
y = vip_data['Retained']

print("3. Splitting the Data (75% Train, 25% Test)...")
# stratify=y ensures the 75/25 balance of VIPs to Downgrades stays identical in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

print("\n" + "="*50)
print("DATA READY FOR MODELING")
print("="*50)
print(f"Training Set created: {X_train.shape[0]:,} rows")
print(f"Testing Set created:  {X_test.shape[0]:,} rows")
print("-" * 50)
print("You may now run Model 1 (Logistic Regression) and Model 2 (Random Forest)!")

1. Finalizing Target Variable (Using your 70% Rule choice)...
2. Isolating the Features...
3. Splitting the Data (75% Train, 25% Test)...

DATA READY FOR MODELING
Training Set created: 60,681 rows
Testing Set created:  20,227 rows
--------------------------------------------------
You may now run Model 1 (Logistic Regression) and Model 2 (Random Forest)!


In [16]:
# going with secnario A for the final model since it provides a more balanced dataset while still defining a meaningful retention threshold.

# --- MODEL 1: TUNED LOGISTIC REGRESSION ---
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

print("1. Building Logistic Regression Pipeline...")
# The Pipeline ensures data is scaled perfectly during cross-validation
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(class_weight='balanced', random_state=42))
])

print("2. Setting up the Hyperparameter Grid...")
# We test 'l1' (Lasso - drops useless features) and 'l2' (Ridge - shrinks weak features)
# 'C' controls how strictly we punish the model for complex formulas
lr_param_grid = {
    'lr__penalty': ['l1', 'l2'],
    'lr__C': [0.01, 0.1, 1, 10],
    'lr__solver': ['liblinear'] # liblinear is required for l1 penalty
}

print("3. Executing Grid Search (Finding the best linear formula)...")
# cv=5 means it tests every combination 5 times to ensure the results aren't just luck
lr_grid = GridSearchCV(lr_pipeline, lr_param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
lr_grid.fit(X_train, y_train)

print("4. Evaluating the Best Logistic Regression Model...")
best_lr = lr_grid.best_estimator_
y_pred_lr = best_lr.predict(X_test)

print("\n" + "="*50)
print("WINNING LOGISTIC REGRESSION CONFIGURATION")
print("="*50)
print(f"Best Parameters: {lr_grid.best_params_}")
print("-" * 50)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))
print("-" * 50)
print("Classification Report:")
print(classification_report(y_test, y_pred_lr))
print("="*50)

1. Building Logistic Regression Pipeline...
2. Setting up the Hyperparameter Grid...
3. Executing Grid Search (Finding the best linear formula)...
4. Evaluating the Best Logistic Regression Model...

WINNING LOGISTIC REGRESSION CONFIGURATION
Best Parameters: {'lr__C': 1, 'lr__penalty': 'l2', 'lr__solver': 'liblinear'}
--------------------------------------------------
Confusion Matrix:
[[ 3913  1249]
 [ 3348 11717]]
--------------------------------------------------
Classification Report:
              precision    recall  f1-score   support

           0       0.54      0.76      0.63      5162
           1       0.90      0.78      0.84     15065

    accuracy                           0.77     20227
   macro avg       0.72      0.77      0.73     20227
weighted avg       0.81      0.77      0.78     20227



In [18]:
# --- MODEL 2: TUNED RANDOM FOREST (FIXED) ---
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

print("1. Setting up the Random Forest Grid...")
# We use a randomized grid to save time while still testing a wide range of architectures
rf_param_dist = {
    'n_estimators': [100, 200, 300],          # How many trees in the forest
    'max_depth': [10, 20, None],              # How deep the "if-then" questions can go
    'min_samples_split': [2, 5, 10],          # Minimum shoppers required to split a group
    'min_samples_leaf': [1, 2, 4]             # Minimum shoppers required to make a final prediction
}

print("2. Executing Randomized Search (Hunting for the best tree structure)...")
print("   (This may take a minute as it builds thousands of trees!)")
rf_base = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)

# THE FIX: The keyword MUST be exactly 'param_distributions'
rf_random = RandomizedSearchCV(rf_base, param_distributions=rf_param_dist, 
                               n_iter=10, cv=3, scoring='f1_macro', random_state=42, n_jobs=-1)
rf_random.fit(X_train, y_train)

print("3. Evaluating the Best Random Forest Model...")
best_rf = rf_random.best_estimator_
y_pred_rf = best_rf.predict(X_test)

print("\n" + "="*50)
print("WINNING RANDOM FOREST CONFIGURATION")
print("="*50)
print(f"Best Parameters: {rf_random.best_params_}")
print("-" * 50)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))
print("-" * 50)
print("Classification Report:")
print(classification_report(y_test, y_pred_rf))
print("="*50)

1. Setting up the Random Forest Grid...
2. Executing Randomized Search (Hunting for the best tree structure)...
   (This may take a minute as it builds thousands of trees!)
3. Evaluating the Best Random Forest Model...

WINNING RANDOM FOREST CONFIGURATION
Best Parameters: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_depth': None}
--------------------------------------------------
Confusion Matrix:
[[ 3254  1908]
 [ 2198 12867]]
--------------------------------------------------
Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.63      0.61      5162
           1       0.87      0.85      0.86     15065

    accuracy                           0.80     20227
   macro avg       0.73      0.74      0.74     20227
weighted avg       0.80      0.80      0.80     20227



In [19]:
# --- STEP 7: ADVANCED THRESHOLD TUNING (PROBABILITY CALIBRATION) ---
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

print("1. Extracting raw probability scores from the Logistic Regression model...")
# predict_proba gets the actual percentage chance, not just the final 0 or 1 guess
y_pred_probs = best_lr.predict_proba(X_test)[:, 1] # Probability of Class 1 (Retained)

print("2. Simulating different Decision Thresholds...")
# We will test shifting the cutoff line from 30% all the way to 70%
thresholds = np.arange(0.30, 0.75, 0.05)

results = []
best_f1 = 0
best_thresh = 0.5

for thresh in thresholds:
    # If the model is > 'thresh' confident they stay, predict 1. Otherwise, predict 0.
    y_pred_custom = (y_pred_probs >= thresh).astype(int)
    
    # Calculate metrics specifically for Class 0 (The Downgraded VIPs)
    f1 = f1_score(y_test, y_pred_custom, pos_label=0)
    recall = recall_score(y_test, y_pred_custom, pos_label=0)
    precision = precision_score(y_test, y_pred_custom, pos_label=0, zero_division=0)
    
    results.append({'Threshold': thresh, 'F1': f1, 'Recall': recall, 'Precision': precision})
    
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

# Display the results neatly
thresh_df = pd.DataFrame(results).round(4)
print("\nThreshold Calibration Results:")
print(thresh_df.to_string(index=False))

print("\n" + "="*50)
print(f"🔥 OPTIMAL THRESHOLD FOUND: {best_thresh:.2f} 🔥")
print(f"This pushes your Class 0 F1-Score up to {best_f1:.4f}!")
print("="*50)

# 3. Apply the winning threshold to see the final matrix
final_predictions = (y_pred_probs >= best_thresh).astype(int)
print("\nFinal Optimized Confusion Matrix:")
print(confusion_matrix(y_test, final_predictions))
print("-" * 50)
print("Final Optimized Classification Report:")
print(classification_report(y_test, final_predictions))

1. Extracting raw probability scores from the Logistic Regression model...
2. Simulating different Decision Thresholds...

Threshold Calibration Results:
 Threshold     F1  Recall  Precision
      0.30 0.5443  0.4655     0.6553
      0.35 0.5830  0.5403     0.6330
      0.40 0.6150  0.6230     0.6071
      0.45 0.6271  0.6922     0.5732
      0.50 0.6300  0.7580     0.5389
      0.55 0.6178  0.8129     0.4983
      0.60 0.5967  0.8601     0.4568
      0.65 0.5693  0.8956     0.4173
      0.70 0.5411  0.9268     0.3820

🔥 OPTIMAL THRESHOLD FOUND: 0.50 🔥
This pushes your Class 0 F1-Score up to 0.6300!

Final Optimized Confusion Matrix:
[[ 3913  1249]
 [ 3348 11717]]
--------------------------------------------------
Final Optimized Classification Report:
              precision    recall  f1-score   support

           0       0.54      0.76      0.63      5162
           1       0.90      0.78      0.84     15065

    accuracy                           0.77     20227
   macro avg       

In [20]:
# --- STEP 8: THE NUCLEAR OPTION (XGBOOST) ---
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

print("1. Preparing XGBoost with balanced sample weights...")
# Instead of SMOTE, we calculate mathematical weights. 
# This tells the algorithm: "If you miss a Class 0 Churner, the penalty is massive."
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

print("2. Training the Extreme Gradient Boosting Model...")
print("   (This is building sequential, error-correcting trees. Give it a moment!)")
xgb_model = XGBClassifier(
    n_estimators=300,         # Total rounds of error-correction
    max_depth=6,              # Deep enough to catch complex behaviors
    learning_rate=0.05,       # Slow learning prevents overfitting
    subsample=0.8,            # Uses 80% of data per tree to prevent memorization
    colsample_bytree=0.8,     # Uses 80% of features per tree to force diversity
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1                 # Use all CPU cores
)

# We pass the sample_weights directly into the fitting process
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)

print("3. Evaluating XGBoost...")
y_pred_xgb = xgb_model.predict(X_test)

print("\n" + "="*50)
print("XGBOOST FINAL EVALUATION")
print("="*50)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))
print("-" * 50)
print("Classification Report:")
print(classification_report(y_test, y_pred_xgb))
print("="*50)

# 4. Feature Importance
print("\nXGBoost Feature Impact (What behaviors actually dictate VIP survival?):")
xgb_importance = pd.DataFrame({
    'Feature': features,
    'Importance_Score': xgb_model.feature_importances_
}).sort_values(by='Importance_Score', ascending=False)

xgb_importance['Importance_Score'] = (xgb_importance['Importance_Score'] * 100).round(2).astype(str) + '%'
print(xgb_importance.to_string(index=False))

1. Preparing XGBoost with balanced sample weights...
2. Training the Extreme Gradient Boosting Model...
   (This is building sequential, error-correcting trees. Give it a moment!)
3. Evaluating XGBoost...

XGBOOST FINAL EVALUATION
Confusion Matrix:
[[ 3966  1196]
 [ 3538 11527]]
--------------------------------------------------
Classification Report:
              precision    recall  f1-score   support

           0       0.53      0.77      0.63      5162
           1       0.91      0.77      0.83     15065

    accuracy                           0.77     20227
   macro avg       0.72      0.77      0.73     20227
weighted avg       0.81      0.77      0.78     20227


XGBoost Feature Impact (What behaviors actually dictate VIP survival?):
                 Feature Importance_Score
            Pre_Monetary           32.93%
           Pre_Frequency           23.79%
        Holiday_Monetary           22.06%
       Holiday_Frequency            9.87%
Holiday_ProductDiversity            

In [21]:
# --- STEP 9: BREAKING THE CEILING WITH SPEND VELOCITY ---
print("1. Extracting November vs. December Spend...")

# Isolate the two months
nov_df = holiday_df[holiday_df['Transaction_Date'].dt.month == 11]
dec_df = holiday_df[holiday_df['Transaction_Date'].dt.month == 12]

# Calculate total spend per month for each customer
nov_spend = nov_df.groupby('CUST_CODE')['SPEND'].sum().compute()
dec_spend = dec_df.groupby('CUST_CODE')['SPEND'].sum().compute()

# Map these back to our vip_data dataframe
vip_data['Nov_Spend'] = vip_data.index.map(nov_spend).fillna(0)
vip_data['Dec_Spend'] = vip_data.index.map(dec_spend).fillna(0)

print("2. Calculating Spend Velocity (Trend)...")
# If Velocity is < 1, their spending was already dying before the new year
vip_data['Holiday_Spend_Velocity'] = vip_data['Dec_Spend'] / vip_data['Nov_Spend'].clip(lower=1)

print("3. Re-Training XGBoost with the new Velocity Feature...")
# Add the new feature to our list
new_features = features + ['Holiday_Spend_Velocity']

X_new = vip_data[new_features]
y_new = vip_data['Retained']

# Re-split the data
X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(X_new, y_new, test_size=0.25, random_state=42, stratify=y_new)

# Re-apply weights
sample_weights_v = compute_sample_weight(class_weight='balanced', y=y_train_v)

# Train XGBoost
xgb_v = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, 
                      subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)
xgb_v.fit(X_train_v, y_train_v, sample_weight=sample_weights_v)

print("\n" + "="*50)
print("VELOCITY-ENHANCED XGBOOST EVALUATION")
print("="*50)
y_pred_v = xgb_v.predict(X_test_v)
print("Classification Report:")
print(classification_report(y_test_v, y_pred_v))
print("="*50)

# Check if the new feature was actually useful
new_importance = pd.DataFrame({
    'Feature': new_features,
    'Importance': xgb_v.feature_importances_
}).sort_values(by='Importance', ascending=False)
print("\nNew Feature Impact:")
print(new_importance.to_string(index=False))

1. Extracting November vs. December Spend...
2. Calculating Spend Velocity (Trend)...
3. Re-Training XGBoost with the new Velocity Feature...

VELOCITY-ENHANCED XGBOOST EVALUATION
Classification Report:
              precision    recall  f1-score   support

           0       0.53      0.77      0.63      5162
           1       0.91      0.76      0.83     15065

    accuracy                           0.77     20227
   macro avg       0.72      0.77      0.73     20227
weighted avg       0.81      0.77      0.78     20227


New Feature Impact:
                 Feature  Importance
            Pre_Monetary    0.321285
           Pre_Frequency    0.241153
        Holiday_Monetary    0.219590
       Holiday_Frequency    0.062863
Holiday_ProductDiversity    0.053774
  Holiday_Spend_Velocity    0.051374
         Holiday_Recency    0.049962


logistic regression model is better by far


In [22]:
# --- STEP 10: OPERATIONALIZING THE MODEL ---
import joblib
import pandas as pd

print("1. Saving the Production Model...")
# We save the 'best_lr' pipeline (which includes the scaler and the model)
joblib.dump(best_lr, 'vip_churn_logistic_model.pkl')
print("   -> Model successfully saved as 'vip_churn_logistic_model.pkl'")

print("\n2. Generating the VIP At-Risk 'Hit List'...")
# Get the exact churn probability (Class 0) for every single VIP in the dataset
all_vips_features = vip_data[features]
vip_data['Churn_Risk_Probability'] = best_lr.predict_proba(all_vips_features)[:, 0]

# Filter down to ONLY the people who are flagged as likely to downgrade (>50% risk)
at_risk_vips = vip_data[vip_data['Churn_Risk_Probability'] >= 0.50].copy()

# Sort the list: We want the highest risk people who spend the MOST money at the very top
at_risk_vips = at_risk_vips.sort_values(by=['Churn_Risk_Probability', 'Holiday_Monetary'], ascending=[False, False])

# Select only the most relevant columns for the Marketing Team
marketing_export = at_risk_vips[['Churn_Risk_Probability', 'Holiday_Monetary', 'Pre_Monetary', 'Holiday_Frequency']]

# Save to CSV
export_filename = 'At_Risk_VIPs_Action_List.csv'
marketing_export.to_csv(export_filename)

print("\n" + "="*50)
print("PROJECT PIPELINE COMPLETE")
print("="*50)
print(f"Total VIPs flagged for Marketing Intervention: {len(marketing_export):,}")
print(f"File saved to your folder as: {export_filename}")
print("="*50)

# Preview the top 5 most critical customers to save
display(marketing_export.head())

1. Saving the Production Model...
   -> Model successfully saved as 'vip_churn_logistic_model.pkl'

2. Generating the VIP At-Risk 'Hit List'...

PROJECT PIPELINE COMPLETE
Total VIPs flagged for Marketing Intervention: 28,862
File saved to your folder as: At_Risk_VIPs_Action_List.csv


,Churn_Risk_Probability,Holiday_Monetary,Pre_Monetary,Holiday_Frequency
CUST_CODE,,,,
CUST0000330990,0.999992,1274.82,2862.42,77
CUST0000369592,0.999960,917.11,1521.26,11
CUST0000168127,0.999950,956.65,1886.75,19
CUST0000735612,0.999939,773.00,700.45,18
CUST0000396614,0.999919,981.15,2072.51,53
